# 03 · LoRA fine-tuning

**What this notebook decides.** The training recipe for condition **B**, and
what to watch while it runs. The foundation weights stay frozen; only adapter
parameters are trained, and only adapter artifacts are saved.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    environment_report,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

## 1 · The recipe

In [ ]:
from qwen_text2sql.config import load_config

config = load_config(ROOT / "configs/qwen35_4b_lora.yaml")

recipe = pd.Series(
    {
        "model_id": config.model.model_id,
        "max_seq_length": config.model.max_seq_length,
        "lora_rank": config.lora.rank,
        "lora_alpha": config.lora.alpha,
        "lora_dropout": config.lora.dropout,
        "target_modules": str(config.lora.target_modules),
        "rslora": config.lora.use_rslora,
        "learning_rate": config.training.learning_rate,
        "epochs": config.training.epochs,
        "effective_batch": config.training.per_device_train_batch_size
        * config.training.gradient_accumulation_steps,
        "completion_only_loss": config.training.completion_only_loss,
        "seed": config.training.seed,
    },
    name="value",
).to_frame()
recipe

Two settings deserve more attention than they usually get.

**`completion_only_loss`.** Loss is computed on the assistant turn only. With it
off, the model spends most of its gradient budget learning to reproduce schema
text — which it is given at inference anyway — and the adapter learns the
dataset's prompt format rather than the task.

**`alpha` relative to `rank`.** PEFT scales the update by `alpha / rank`. Holding
`alpha` fixed while sweeping `rank` therefore changes the effective learning rate
at the same time as the capacity, and the rank ablation in
[`08_adapter_rank_ablation`](08_adapter_rank_ablation.ipynb) would be
confounded. The check below makes that explicit.

In [ ]:
scale = config.lora.alpha / config.lora.rank
print(f"alpha/rank scaling = {scale:.2f}")
print(
    "When sweeping rank, either hold alpha/rank constant by scaling alpha with rank,\n"
    "or enable use_rslora (which rescales by 1/sqrt(rank)). Otherwise capacity and\n"
    "effective step size move together and the ablation cannot attribute the result."
)

## 2 · Adapter size scales linearly in rank

A LoRA adapter adds `rank x (d_in + d_out)` parameters per targeted projection.
Every other term is fixed by the architecture, so adapter size is exactly linear
in rank. That relation needs no architecture constants and no download:

In [ ]:
RANKS = (4, 8, 16, 32, 64)
scaling = pd.DataFrame(
    {"lora_rank": RANKS},
).assign(
    relative_adapter_size=lambda frame: frame["lora_rank"] / config.lora.rank,
    alpha_for_fixed_scaling=lambda frame: frame["lora_rank"] * scale,
)
scaling

The exact trainable-parameter count is architecture-dependent, so rather than
hard-coding dimensions here, `scripts/train_adapter.py` reports the real count at
startup via PEFT's `print_trainable_parameters()`. Take the number from the run
log; do not estimate it in a notebook.

## 3 · Running the training

```bash
poetry run python scripts/train_adapter.py \
  --config configs/qwen35_4b_lora.yaml \
  --train-data data/processed/bird_train.jsonl \
  --validation-data data/processed/bird_validation.jsonl
```

**What to watch, in priority order.**

1. **Validation loss, not training loss.** Training loss falls monotonically for
   a well-tuned LoRA whether or not it is learning anything transferable.
2. **The gap between them.** Widening after the first epoch on a few thousand
   examples is the expected regime for adapters — it is a reason to stop early,
   not to raise the rank.
3. **Trainable parameter count at startup.** If it is ~100% of the model, the
   base weights are not frozen and this is no longer a LoRA experiment.

Validation loss is a *stopping* signal only. Selecting a checkpoint on validation
execution accuracy and then reporting that accuracy would be tuning on the
evaluation set, which rule 1 forbids.

In [ ]:
adapter_dir = ROOT / config.training.output_dir
print(f"Adapter output directory: {adapter_dir}")
print(f"Exists: {adapter_dir.exists()}")
print(
    "\nOnly PEFT adapter artifacts are written here — adapter_model.safetensors and\n"
    "adapter_config.json, typically tens of megabytes. If you see multi-gigabyte\n"
    "files, full model weights are being saved and .gitignore is your last defence."
)

## Reading this recipe

**What it shows.** The exact, reproducible recipe for condition B and the two
settings most likely to invalidate the downstream ablation.

**What it does not show.** No result. Fine-tuning is evaluated in
[`05_execution_evaluation`](05_execution_evaluation.ipynb), against the baseline
from [`02_foundation_model_baseline`](02_foundation_model_baseline.ipynb).

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.